# Column entropy and clicks

This notebook computes the Shannon entropy of each alignment column in numpy, draws it as a bar track, highlights the columns above a slider's threshold, and prints the column a click lands on.

In [ ]:
from pathlib import Path

import numpy as np

DATA = Path("../../examples/data")
msa = (DATA / "globin.aln").read_text()

rows = {}
for record in msa.split(">")[1:]:
    header, *lines = record.splitlines()
    rows[header.split()[0]] = "".join(lines)
cells = np.array([list(row) for row in rows.values()])
cells.shape

The entropy counts a column's residues and ignores its gaps, so a column holding one residue scores 0 bits and twenty equally frequent amino acids score log2(20), about 4.32.

In [ ]:
def column_entropy(column):
    residues = column[column != "-"]
    if residues.size == 0:
        return 0.0
    _, counts = np.unique(residues, return_counts=True)
    p = counts / residues.size
    return float(-(p * np.log2(p)).sum())


entropy = np.array([column_entropy(c) for c in cells.T])

A numpy array works as a bar track's `values`. `highlights` takes 1-based inclusive `{start, end}` column ranges, so the slider's handler joins consecutive columns into one range.

In [ ]:
import ipywidgets as widgets

from msaview import MSAView

view = MSAView(
    msa=msa,
    tree=DATA / "globin.nh",
    column_tracks=[
        {
            "id": "entropy",
            "name": "Shannon entropy (bits)",
            "kind": "bar",
            "values": entropy,
            "max": float(np.log2(20)),
        }
    ],
    hide_header=True,
    height=360,
)


def highlight_above(threshold):
    columns = np.flatnonzero(entropy > threshold) + 1
    breaks = np.flatnonzero(np.diff(columns) > 1)
    starts = np.r_[columns[:1], columns[breaks + 1]]
    ends = np.r_[columns[breaks], columns[-1:]]
    view.highlights = [
        {"start": s, "end": e} for s, e in zip(starts.tolist(), ends.tolist())
    ]


printed = widgets.Output()


def print_column(cell):
    printed.clear_output()
    if cell is None:
        return
    with printed:
        print(f"column {cell['column']}, clicked on {cell.get('row')}")
        for name, row in rows.items():
            print(f"  {name:<18} {row[cell['column'] - 1]}")


slider = widgets.FloatSlider(value=2, min=0, max=4.3, step=0.1, description="bits above")
slider.observe(lambda change: highlight_above(change["new"]), "value")
view.observe(lambda change: print_column(change["new"]), "clicked")
highlight_above(slider.value)
widgets.VBox([slider, view, printed])

A click sets `view.clicked` to `{column, row, residue, letter}`, and `view.viewport` holds the columns on screen once a scroll or zoom settles. The cell below calls the handler with the value a click on human beta globin's distal histidine sets.

In [ ]:
print_column({"column": 82, "row": "Human_beta", "residue": 64, "letter": "H"})